# Sequence profile algorithm

In [1]:
###imports
import numpy as np
import os

cwd = os.getcwd()
parent = os.path.dirname(cwd)
grandparent = os.path.dirname(parent)

data_dir =  f"{parent}/data/"

In [31]:
###load protein sequences 
seqs = []
#location = []
encoding = {}

alphabet_file = data_dir + "alphabet"
alphabet = np.loadtxt(alphabet_file, dtype=str)

for i, AA in enumerate(alphabet):
    encoding[AA] = i



with open(data_dir + "signalpeptide.csv") as f:
    #print(f.read().split())
    for i,line in enumerate(f.read().split()):
        #dont include the first line
        if i > 1:
            current_sequence = line.split(",")[0]
            if i == 2:
                sequence_length = len(current_sequence)
            if len(current_sequence) == sequence_length:
                enc_sequence = []
                for AA in current_sequence:
                    enc_sequence.append( encoding[AA] )
                    #encode the sequence to be between 1 and 20 (corresponding to an AA)
                seqs.append(enc_sequence)


cyt_seqs = []
with open(data_dir + "cytosolpeptide.csv") as f:
    for i,line in enumerate(f.read().split()):
        #dont include the first line
        if i > 1:
            current_sequence = line.split(",")[0]
            if i == 2:
                cyt_sequence_length = len(current_sequence)
            if len(current_sequence) == cyt_sequence_length:
                enc_sequence = []
                for AA in current_sequence:
                    enc_sequence.append( encoding[AA] )
                    #encode the sequence to be between 1 and 20 (corresponding to an AA)
                cyt_seqs.append(enc_sequence)

#output:
## two lists, 
### one with 30-mer peptides
### one with translocation

In [34]:
def Initialize_model(seqs):
    ### Initialize architecture
    sequence_length = len(seqs[0])

    #sequence always starts with M, so initial propability for M is 1, and everything else is 0
    initial_prop = [0 if p != "M" else 1 for p in alphabet ]
    #print(initial_prop)

    #####states
    states = []
    # all states has an P-I (also last state)
    for i in range(1, sequence_length+1):
        
        states.append(f"P{i}")
        states.append(f"I{i}")
        #states.append(f"D{i+1}")
    

    ### Emissions (each row is a state and each column is an emission from the alphabet)
    #initial emission propabilty (equal propabilty for all AA)
    p_initial_emission = [1/len(alphabet)]*len(alphabet)
    #d_emission = [0] *len(alphabet) + [1] # always ""
    i_initial_emission = [1/len(alphabet)]*len(alphabet)

    emission = np.array([
        p_initial_emission if state.startswith("P")
        #else d_emission if state.startswith("D")
        else i_initial_emission #if state.startswith("I")
        #else d_emission  # end state
        for state in states
    ])


    ###initial transmission propability
    transmission = np.zeros ( (len(states), len(states)) )

    for i, s in enumerate(states):
        #if it is not the last 2 states
        if i != len(states) -1:
            if s[0] == "P":
                #check add transmission to I state
                transmission[i,i+1] = 1
                # add transmission to all future P states
                for j in range(i+1, len(states)):
                    if states[j].startswith("P"):
                        transmission[i,j] = 1
            elif s[0] == "I":
                transmission[i,i] = 1
                transmission[i,i+1] = 1
        else:
            transmission[i,i] = 1

    initial_transmission = transmission

    return initial_prop, transmission, emission, initial_transmission, sequence_length, states

### Functions

In [56]:

def calculate_alpha(input_encode, states, transmission, emission, initial_transmission, weight = 1):
    
    alpha = np.zeros(shape=(len(states), len(input_encode)))
    alpha[0][0] = 1

    c = np.zeros(len(input_encode))
    c[0] = np.sum(alpha[:,0])
    alpha[:,0] /= c[0]
        
    #make first row (shape = (states, sequence length))
    

    # main loop
    #for each position in sequence
    for i in range(1, len(input_encode)):
        #for each state, j at new position i
        for j in range(0, len(states)):

            _sum = 0
            #sum over all states, k of old position i
            for k in range(0, len(states)):
                if initial_transmission[k,j] != 0:
                    if states[j].startswith("D"):
                        #Deletion state does not move forwards in the sequence and has no emision
                        _sum += alpha[k][i] * transmission[k][j]
                    else:
                        _sum += alpha[k][i-1]*transmission[k][j]*emission[j][input_encode[i]]
            
            # store prob
            alpha[j][i] = _sum * weight
        c[i] = np.sum(alpha[:,i])


        #print("position", i)
        #print("symbol", input_encode[i])

        #reachable = np.where(alpha[:,i-1] > 0)[0]

        #print("reachable states:", reachable)

        #for j in reachable[:10]:
        #    print(
        #        j,
        #        states[j],
        #        emission[j,input_encode[i]]
        #    )
        if c[i] == 0:
            print("ZERO COLUMN", i)
            print("symbol", input_encode[i])
            break

        if c[i] > 0:
            alpha[:,i] /= c[i]
    return alpha, c

def calculate_beta(input_encode, states, transmission, emission, initial_transmission, c,weight = 1):

    
    beta = np.zeros(shape=(len(states), len(input_encode)))
    beta[:,len(input_encode)-1] = 1
    beta[:,len(input_encode)-1] /= c[len(input_encode)-1]


    # main loop
    # for each position in sequence starting from second to last
    for i in range(len(input_encode)-2, -1, -1):
        #for each state in current j
        for j in range(0, len(states)):

            _sum = 0
            # for each state in the next position (i+1)
            for k in range(0, len(states)):
                #if transmission is possible
                if initial_transmission[j,k] != 0:
                    _sum += emission[k][input_encode[i+1]] * beta[k][i+1] *transmission[j][k]
            
            # store prob
            beta[j][i] = _sum * weight
        beta[:,i] /= c[i]

    return beta

def calculate_gamma(input_encode, states, alpha, beta):
    gamma = np.zeros(shape=(len(states), len(input_encode)))

    #for each position t
    for t in range(len(input_encode)):
        #calculate alpha[i]*beta[i] for the entire column
        for i in range(len(states)):
            gamma[i][t] = alpha[i][t] * beta[i][t] #not normalized yet!
            #print(gamma[i][t])
        #normalize entire column by the sum of the column
        #print(gamma[:][t])
        c_sum = sum(gamma[:,t])
        for i in range(len(states)):
            if c_sum != 0:
                gamma[i][t] = gamma[i][t] / c_sum
    return gamma

def calculate_xi(input_encode, states, alpha, beta, initial_transmission, sequence_length, transmission, emission):
    #Initialize xi (state from, state to) (we will sum over all positions (not last position) to imidiatly calculate the sum used in a_new)
    xi = np.zeros(shape= (len(states), len(states), sequence_length))

    #for each from state
    for i in range(len(states)):
        #for each to state
        for j in range(len(states)):
            #if transition from i to j is possible (not 0 at the beginning)
            if initial_transmission[i,j] != 0:
                #_sum = 0
                for t in range(sequence_length-1):
                    xi[i,j,t]= alpha[i][t] * transmission[i,j]* emission[j,input_encode[t+1]] * beta[j,t+1]
                #xi_psum[i,j] = _sum/aT_sum

    #renormalize
    #xi[:,:,t] /= np.sum(xi[:,:,t])
    denom = np.sum(xi[:,:,t])
    if denom > 0:
        xi[:,:,t] /= denom
    ## returns xi summed over all t
    return np.sum(xi, axis=2)

def calculate_new_transmission(gamma, xi_psum, initial_transmission, states):
    
    gamma_sum = np.zeros(shape = (len(states)))
    for i in range(len(states)):
        gamma_sum[i] = np.sum(gamma[i,:-1])
        
    trans_new = initial_transmission.copy()
    for i in range(len(states)):
        for j in range(len(states)):
            if initial_transmission[i,j] != 0:
                #include max to avoid division by 0

                #For multiple sequences is is the sum of xi_psum for all sequences divided by the sum of gamma_sum[i] for all sequences 
                trans_new[i,j] = xi_psum[i,j]/max(gamma_sum[i],0.00000001)
                
    for i in range(len(states)):
        row_sum = np.sum(trans_new[i,:])
        if row_sum > 0:
            trans_new[i,:] /= row_sum
    return trans_new

def calculate_new_emission(input_encode, states, gamma, emission, sequence_length):
    #the positions of the states that are not D and end
    pi_states = [i for i,s in enumerate(states) if s.startswith(("P","I")) ]
    #print(alphabet)
    emis_new = emission.copy()

    #for state i that is eiter P or I state (don't update deletions and end)
    for i in pi_states:
        for a in alphabet:
            _sum = 0
            #sum of all observations that are equal to current symbol
            for t in range(sequence_length):
                if input_encode[t] == encoding[a]:
                    _sum += gamma[i,t]
            
            emis_new[i,encoding[a]] = _sum / max(np.sum(gamma[i,:]), 0.0001)
    return emis_new


def viterbi(input_encode, states, initial_prob, transmission, emission_probs):
    
    ## Initialize
    delta = np.zeros(shape=(len(states), len(input_encode)))
    arrows = np.ndarray(shape=(len(states), len(input_encode)), dtype=object)
    # initial conditions
    delta[0][0] = 1
    for i in range(0, len(states)):
        arrows[i][0] = 0

    for i in range(1, len(input_encode)):
        #the state we are about to transition to
        for j in range(0, len(states)):
            
            max_arrow_prob = -np.inf # A very low negative number
            max_arrow_prob_state = -1
            
            #the state that we are comming from
            for k in range(0, len(states)):
                
                # arrow_prob is the probability of ending in the state j from the state k
                # [state we are in, state we wish to transition to]
                arrow_prob = transmission[k,j]+ delta[k][i-1]
                
                if arrow_prob > max_arrow_prob: 
                    max_arrow_prob = arrow_prob
                    max_arrow_prob_state = k
                
            # store prob
            delta[j][i] =  emission_probs[j][input_encode[i]]  + max_arrow_prob

            # store arrow
            arrows[j][i] = max_arrow_prob_state

    max_state = np.argmax(delta[:, -1]) # Find the index of the max value in the last column of delta
    max_value = delta[max_state, -1] # Find the max value in the last column of delta

    return max_value

## Main Loop

In [59]:
initial_prop, transmission, emission, initial_transmission, sequence_length, states = Initialize_model(seqs)
iterations = 2

#mit_test_seq = 
#cyt_test_seq = 

#split data into chunks
chunk_size = 100
test_size = 1
n_chunks = (len(seqs)-test_size)//chunk_size
chunk_count = 0

for i in range(iterations):
    #iterate over all sequences
    xi_seq_sum = np.zeros(shape= (len(states), len(states)))
    gamma_seq_sum = np.zeros(shape=(len(states), sequence_length))

    #calculate current chunk
    c_start = (chunk_count % n_chunks) * chunk_size
    c_end = min(c_start+chunk_size, len(seqs)-test_size)
    print(np.sum(emission[:,17] == 0))
    
    for s in seqs[c_start: c_end]:
        input = s
        #print("calculate alpha")
        alpha, scaling = calculate_alpha(input, states, transmission, emission, initial_transmission)
        #print("b")
        beta =calculate_beta(input, states, transmission, emission, initial_transmission, scaling)
        gamma = calculate_gamma(input, states, alpha, beta)
        xi_psum = calculate_xi(input, states, alpha, beta, initial_transmission, sequence_length, transmission, emission)
        xi_seq_sum += xi_psum
        gamma_seq_sum += gamma

    
    transmission = calculate_new_transmission(gamma_seq_sum, xi_seq_sum, initial_transmission, states)
    #print(np.sum(transmission, axis=1)[0])
    emission = calculate_new_emission(input, states, gamma_seq_sum, emission, sequence_length)
    print(np.sum(emission[:,17] == 0))
    #test test-data
    for ts in seqs[-test_size:]:
        #print(ts)
        alpha_ts, _ = calculate_alpha(ts, states, transmission, emission, initial_transmission)
        #print(np.max(alpha_ts))
        #print(np.min(alpha_ts[alpha_ts > 0]))
        prop_ts = np.sum(alpha_ts[:,-1] )
        #log_prop_ts = np.log10(prop_ts)
        #print(prop_ts)
        
    
    
    print(f"iteration {i}, ")

    



0
1
ZERO COLUMN 2
symbol 2
iteration 0, 
1
ZERO COLUMN 21
symbol 16


C:\Users\jolle\AppData\Local\Temp\ipykernel_21328\2851228787.py:61: RuntimeWarning: divide by zero encountered in divide
  beta[:,len(input_encode)-1] /= c[len(input_encode)-1]
C:\Users\jolle\AppData\Local\Temp\ipykernel_21328\2851228787.py:75: RuntimeWarning: invalid value encountered in scalar multiply
  _sum += emission[k][input_encode[i+1]] * beta[k][i+1] *transmission[j][k]
C:\Users\jolle\AppData\Local\Temp\ipykernel_21328\2851228787.py:90: RuntimeWarning: invalid value encountered in scalar multiply
  gamma[i][t] = alpha[i][t] * beta[i][t] #not normalized yet!
C:\Users\jolle\AppData\Local\Temp\ipykernel_21328\2851228787.py:112: RuntimeWarning: invalid value encountered in scalar multiply
  xi[i,j,t]= alpha[i][t] * transmission[i,j]* emission[j,input_encode[t+1]] * beta[j,t+1]


ZERO COLUMN 26
symbol 8
ZERO COLUMN 4
symbol 8
ZERO COLUMN 6
symbol 2
ZERO COLUMN 15
symbol 8
ZERO COLUMN 2
symbol 16
ZERO COLUMN 6
symbol 8
ZERO COLUMN 3
symbol 8
ZERO COLUMN 27
symbol 8
ZERO COLUMN 4
symbol 16
ZERO COLUMN 19
symbol 16
ZERO COLUMN 14
symbol 16
ZERO COLUMN 9
symbol 16
ZERO COLUMN 18
symbol 8
ZERO COLUMN 25
symbol 8
ZERO COLUMN 17
symbol 2
ZERO COLUMN 24
symbol 16
ZERO COLUMN 37
symbol 16
ZERO COLUMN 5
symbol 2
ZERO COLUMN 2
symbol 2
ZERO COLUMN 8
symbol 16
ZERO COLUMN 6
symbol 16
ZERO COLUMN 3
symbol 16
ZERO COLUMN 2
symbol 2
ZERO COLUMN 28
symbol 16
ZERO COLUMN 3
symbol 8
ZERO COLUMN 5
symbol 2
ZERO COLUMN 5
symbol 2
ZERO COLUMN 11
symbol 16
ZERO COLUMN 36
symbol 8
ZERO COLUMN 10
symbol 2
ZERO COLUMN 40
symbol 16
ZERO COLUMN 26
symbol 16
ZERO COLUMN 18
symbol 16
ZERO COLUMN 7
symbol 16
ZERO COLUMN 7
symbol 2
ZERO COLUMN 8
symbol 2
ZERO COLUMN 2
symbol 8
ZERO COLUMN 24
symbol 2
ZERO COLUMN 1
symbol 16
ZERO COLUMN 7
symbol 8
ZERO COLUMN 21
symbol 16
ZERO COLUMN 42
symbo